# Kafka Streaming — Databricks Integration

Stream real-time synthetic data from Kafka into Databricks using Structured Streaming.

**9 use cases available:**

| Use Case | Topic | Category |
|---|---|---|
| `fraud_detection` | `streaming-fraud-transactions` | Core |
| `telemetry` | `streaming-device-telemetry` | Core |
| `web_traffic` | `streaming-web-traffic` | Core |
| `student_enrollment` | `streaming-student-enrollment` | SLED |
| `grant_budget` | `streaming-grant-budget` | SLED |
| `citizen_services` | `streaming-citizen-services` | SLED |
| `k12_early_warning` | `streaming-k12-early-warning` | SLED |
| `procurement` | `streaming-procurement` | SLED |
| `case_management` | `streaming-case-management` | SLED |

In [ ]:
%run "./_config"

---
## Start Generators

In [ ]:
ALL_USE_CASES = [
    "fraud_detection", "telemetry", "web_traffic",
    "student_enrollment", "grant_budget", "citizen_services",
    "k12_early_warning", "procurement", "case_management",
]

for use_case in ALL_USE_CASES:
    resp = requests.post(f"{API_URL}/kafka/generators/start", headers=HEADERS, json={
        "use_case": use_case,
        "rows_per_batch": 100,
        "batch_interval_seconds": 1.0,
        "timeout_minutes": 10,
    })
    data = resp.json()
    print(f"{use_case}: id={data.get('generator_id')} status={data.get('status')}")

generators = requests.get(f"{API_URL}/kafka/generators", headers=HEADERS).json()
print(f"\n{len(generators)} generator(s) running")

---
## Core — Fraud Detection

In [ ]:
fraud_schema = StructType([
    StructField("transaction_id", StringType()),
    StructField("user_id", IntegerType()),
    StructField("merchant_id", IntegerType()),
    StructField("amount", DecimalType(10, 2)),
    StructField("currency", StringType()),
    StructField("merchant_category", StringType()),
    StructField("payment_method", StringType()),
    StructField("ip_address", StringType()),
    StructField("device_id", StringType()),
    StructField("latitude", DecimalType(8, 5)),
    StructField("longitude", DecimalType(9, 5)),
    StructField("card_type", StringType()),
    StructField("event_timestamp", StringType()),
])

df = kafka_to_delta("streaming-fraud-transactions", fraud_schema, "kafka_fraud_transactions")
display(df)


---
## Core — Telemetry

In [ ]:
telemetry_schema = StructType([
    StructField("event_id", StringType()),
    StructField("device_id", StringType()),
    StructField("device_type", StringType()),
    StructField("reading_value", DecimalType(10, 4)),
    StructField("unit", StringType()),
    StructField("battery_level", DecimalType(5, 2)),
    StructField("signal_strength_dbm", IntegerType()),
    StructField("firmware_version", StringType()),
    StructField("facility_id", IntegerType()),
    StructField("latitude", DecimalType(8, 5)),
    StructField("longitude", DecimalType(9, 5)),
    StructField("anomaly_flag", BooleanType()),
    StructField("event_timestamp", StringType()),
])

df = kafka_to_delta("streaming-device-telemetry", telemetry_schema, "kafka_device_telemetry")
display(df)


---
## Core — Web Traffic

In [ ]:
web_schema = StructType([
    StructField("event_id", StringType()),
    StructField("session_id", StringType()),
    StructField("user_id", IntegerType()),
    StructField("page_url", StringType()),
    StructField("referrer", StringType()),
    StructField("user_agent", StringType()),
    StructField("ip_address", StringType()),
    StructField("country", StringType()),
    StructField("device_type", StringType()),
    StructField("action", StringType()),
    StructField("duration_ms", IntegerType()),
    StructField("http_status", IntegerType()),
    StructField("event_timestamp", StringType()),
])

df = kafka_to_delta("streaming-web-traffic", web_schema, "kafka_web_traffic")
display(df)


---
## SLED — Student Enrollment

In [ ]:
enrollment_schema = StructType([
    StructField("event_id", StringType()), StructField("student_id", StringType()),
    StructField("course_id", StringType()), StructField("action", StringType()),
    StructField("semester", StringType()), StructField("department", StringType()),
    StructField("grade", StringType()), StructField("credits", IntegerType()),
    StructField("campus", StringType()), StructField("gpa_impact", DecimalType(3, 1)),
    StructField("event_timestamp", StringType()),
])

df = kafka_to_delta("streaming-student-enrollment", enrollment_schema, "kafka_student_enrollment")
display(df)


---
## SLED — Grant & Budget

In [ ]:
budget_schema = StructType([
    StructField("transaction_id", StringType()), StructField("fund_source_id", StringType()),
    StructField("agency_id", StringType()), StructField("program_id", StringType()),
    StructField("vendor_id", StringType()), StructField("transaction_type", StringType()),
    StructField("amount", DecimalType(10, 2)), StructField("fund_category", StringType()),
    StructField("fiscal_year", StringType()), StructField("quarter", StringType()),
    StructField("cost_center", StringType()), StructField("account_code", StringType()),
    StructField("description", StringType()), StructField("event_timestamp", StringType()),
])

df = kafka_to_delta("streaming-grant-budget", budget_schema, "kafka_grant_budget")
display(df)


---
## SLED — Citizen Services (311)

In [ ]:
citizen_schema = StructType([
    StructField("request_id", StringType()), StructField("citizen_id", StringType()),
    StructField("request_type", StringType()), StructField("department", StringType()),
    StructField("status", StringType()), StructField("priority", StringType()),
    StructField("district", IntegerType()), StructField("asset_id", StringType()),
    StructField("latitude", DecimalType(8, 5)), StructField("longitude", DecimalType(9, 5)),
    StructField("response_time_hours", IntegerType()), StructField("satisfaction_rating", IntegerType()),
    StructField("event_timestamp", StringType()),
])

df = kafka_to_delta("streaming-citizen-services", citizen_schema, "kafka_citizen_services")
display(df)


---
## SLED — K-12 Early Warning

In [ ]:
k12_schema = StructType([
    StructField("event_id", StringType()), StructField("student_id", StringType()),
    StructField("school_id", StringType()), StructField("event_type", StringType()),
    StructField("grade_level", IntegerType()), StructField("teacher_id", StringType()),
    StructField("risk_score", DecimalType(5, 2)), StructField("attendance_rate", DecimalType(5, 2)),
    StructField("gpa", DecimalType(3, 2)), StructField("behavior_incidents_ytd", IntegerType()),
    StructField("intervention_type", StringType()), StructField("school_type", StringType()),
    StructField("free_reduced_lunch", BooleanType()), StructField("english_learner", BooleanType()),
    StructField("special_education", BooleanType()), StructField("event_timestamp", StringType()),
])

df = kafka_to_delta("streaming-k12-early-warning", k12_schema, "kafka_k12_early_warning")
display(df)


---
## SLED — Procurement

In [ ]:
procurement_schema = StructType([
    StructField("event_id", StringType()), StructField("agency_id", StringType()),
    StructField("vendor_id", StringType()), StructField("event_type", StringType()),
    StructField("contract_id", StringType()), StructField("amount", DecimalType(12, 2)),
    StructField("procurement_method", StringType()), StructField("commodity_code", StringType()),
    StructField("category", StringType()), StructField("minority_owned", BooleanType()),
    StructField("small_business", BooleanType()), StructField("local_vendor", BooleanType()),
    StructField("contract_duration_months", IntegerType()), StructField("payment_terms", StringType()),
    StructField("event_timestamp", StringType()),
])

df = kafka_to_delta("streaming-procurement", procurement_schema, "kafka_procurement")
display(df)


---
## SLED — Case Management (HHS)

In [ ]:
case_schema = StructType([
    StructField("event_id", StringType()), StructField("client_id", StringType()),
    StructField("case_id", StringType()), StructField("caseworker_id", StringType()),
    StructField("event_type", StringType()), StructField("program", StringType()),
    StructField("agency_id", StringType()), StructField("benefit_amount", DecimalType(10, 2)),
    StructField("household_size", IntegerType()), StructField("income_bracket", StringType()),
    StructField("county", StringType()), StructField("determination", StringType()),
    StructField("referral_source", StringType()), StructField("priority", StringType()),
    StructField("event_timestamp", StringType()),
])

df = kafka_to_delta("streaming-case-management", case_schema, "kafka_case_management")
display(df)


---
## Cleanup

In [ ]:
generators = requests.get(f"{API_URL}/kafka/generators", headers=HEADERS).json()
for g in generators:
    if g["status"] == "running":
        requests.post(f"{API_URL}/kafka/generators/{g['generator_id']}/stop", headers=HEADERS)
        print(f"Stopped {g['use_case']} generator {g['generator_id']}")

cleanup = requests.delete(f"{API_URL}/kafka/generators/cleanup", headers=HEADERS).json()
print(f"Cleaned up {cleanup.get('removed', 0)} generator(s)")